# 01 Multinomial Logit: Random Utility for Three Travel Modes

## Program Overview

This program teaches MNL with a compact SwissMetro-style example. It uses the same conceptual structure as the MOOC Biogeme examples: define parameters, build utility functions, compute choice probabilities, and compare predictions with observed choices.

## Connection to Part 2 slides

Use after Part 2 slides on the Logit model, MNL, and random utility theory.

## Overall Task AI Prompt (English)

Use this notebook as a programming lesson for discrete choice modeling. Before
running each code cell, read the prompt and explain what theoretical concept the
code implements. After running the cell, use the interpretation cell to connect
the numerical result back to utility, probability, likelihood, estimation, or
model validation.

## Learning Objectives

- Define the decision maker, alternatives, choice variable, and explanatory attributes.
- Translate random utility theory into executable Python code.
- Interpret model probabilities, coefficients, likelihood values, and validation results.
- Distinguish behavioral explanation from prediction accuracy.


## AI Prompt for This Code Cell

Create a SwissMetro-style mode choice data set with three alternatives: car, train, and SwissMetro.

In [1]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    "person_id": np.arange(1, 9),
    "choice": ["SM", "TRAIN", "CAR", "SM", "TRAIN", "CAR", "SM", "TRAIN"],
    "car_time": [70, 95, 60, 80, 100, 65, 75, 90],
    "train_time": [65, 75, 85, 70, 72, 82, 68, 76],
    "sm_time": [45, 55, 50, 48, 58, 52, 47, 54],
    "car_cost": [35, 42, 30, 38, 44, 32, 36, 41],
    "train_cost": [28, 33, 31, 29, 35, 30, 27, 34],
    "sm_cost": [32, 36, 34, 33, 39, 35, 31, 37],
})

data

,person_id,choice,car_time,train_time,sm_time,car_cost,train_cost,sm_cost
0,1,SM,70,65,45,35,28,32
1,2,TRAIN,95,75,55,42,33,36
2,3,CAR,60,85,50,30,31,34
3,4,SM,80,70,48,38,29,33
4,5,TRAIN,100,72,58,44,35,39
5,6,CAR,65,82,52,32,30,35
6,7,SM,75,68,47,36,27,31
7,8,TRAIN,90,76,54,41,34,37


## Output Interpretation

The output shows 8 observations with three observed choices for SwissMetro, three for train, and two for car. This is a compact version of a SwissMetro-style mode-choice problem: each traveler has time and cost attributes for all three alternatives, not only for the chosen alternative. That full choice-set structure is essential for MNL because the probability of each chosen mode is determined by comparing its utility with the utilities of the other available modes. In class, ask students to identify the observation unit, the choice variable, and the alternative-specific attributes before moving to the model equations.


## AI Prompt for This Code Cell

Define systematic utility functions for car, train, and SwissMetro. Use alternative-specific constants and generic time and cost coefficients.

In [2]:
params = {
    "ASC_CAR": 0.00,
    "ASC_TRAIN": 0.25,
    "ASC_SM": 0.55,
    "B_TIME": -0.035,
    "B_COST": -0.060,
}

def compute_utilities(df, params):
    utilities = pd.DataFrame(index=df.index)
    utilities["CAR"] = (
        params["ASC_CAR"]
        + params["B_TIME"] * df["car_time"]
        + params["B_COST"] * df["car_cost"]
    )
    utilities["TRAIN"] = (
        params["ASC_TRAIN"]
        + params["B_TIME"] * df["train_time"]
        + params["B_COST"] * df["train_cost"]
    )
    utilities["SM"] = (
        params["ASC_SM"]
        + params["B_TIME"] * df["sm_time"]
        + params["B_COST"] * df["sm_cost"]
    )
    return utilities

utilities = compute_utilities(data, params)
utilities

,CAR,TRAIN,SM
0,-4.550,-3.705,-2.945
1,-5.845,-4.355,-3.535
2,-3.900,-4.585,-3.240
3,-5.080,-3.940,-3.110
4,-6.140,-4.370,-3.820
5,-4.195,-4.420,-3.370
6,-4.785,-3.750,-2.955
7,-5.610,-4.450,-3.560


## Output Interpretation

The average systematic utilities are approximately CAR = -5.013, TRAIN = -4.197, and SM = -3.317. SwissMetro has the highest average utility because it has shorter travel times in the teaching data and a positive alternative-specific constant (`ASC_SM = 0.55`). Car has the lowest average utility because its travel times and costs are generally higher and its constant is normalized to zero. The negative utilities are not a problem: only utility differences matter in discrete choice models. The behavioral reading is that time and cost penalties reduce utility, while the constants shift the average attractiveness of each mode.


## AI Prompt for This Code Cell

Convert utilities into Multinomial Logit probabilities using the softmax formula.

In [3]:
def mnl_probabilities(utilities):
    centered = utilities.sub(utilities.max(axis=1), axis=0)
    exp_u = np.exp(centered)
    return exp_u.div(exp_u.sum(axis=1), axis=0)

probabilities = mnl_probabilities(utilities)
probabilities.round(3)

,CAR,TRAIN,SM
0,0.120,0.280,0.599
1,0.064,0.286,0.649
2,0.291,0.147,0.563
3,0.089,0.277,0.635
4,0.059,0.344,0.597
5,0.245,0.196,0.559
6,0.100,0.280,0.620
7,0.084,0.267,0.650


## Output Interpretation

The average predicted probabilities are approximately CAR = 0.131, TRAIN = 0.260, and SM = 0.609. This means the model strongly favors SwissMetro on average. Because probabilities are computed with the MNL softmax formula, a higher utility becomes a larger exponentiated utility share within each row. The result also reveals a teaching issue: the current illustrative parameters are not estimated from the 8 observations, so they impose a strong prior preference for SwissMetro. Students should connect this to W3: estimation is needed when we want parameter values that best explain the observed choices rather than values chosen by the instructor.


## AI Prompt for This Code Cell

Compare the predicted mode with the observed mode and compute a simple prediction accuracy.

In [4]:
prediction = probabilities.idxmax(axis=1)
comparison = data[["person_id", "choice"]].copy()
comparison["predicted_choice"] = prediction
comparison["correct"] = comparison["choice"] == comparison["predicted_choice"]

accuracy = comparison["correct"].mean()
print(comparison)
print(f"Prediction accuracy: {accuracy:.2%}")

   person_id choice predicted_choice  correct
0          1     SM               SM     True
1          2  TRAIN               SM    False
2          3    CAR               SM    False
3          4     SM               SM     True
4          5  TRAIN               SM    False
5          6    CAR               SM    False
6          7     SM               SM     True
7          8  TRAIN               SM    False
Prediction accuracy: 37.50%


## Output Interpretation

The model predicts SwissMetro for all 8 observations, so it correctly predicts the 3 SwissMetro choices but misses all train and car choices. The resulting accuracy is 37.50%. This is not a coding failure; it is an important modeling lesson. The chosen parameters generate average probabilities that heavily favor SwissMetro (`0.609`), so the maximum-probability rule always selects SwissMetro. This demonstrates why prediction accuracy must be interpreted together with parameter estimation and behavioral plausibility. For a real model, we would estimate the constants and time/cost coefficients so that train and car choices are also represented by the fitted probabilities.
